# 02 — Baselines: el piso a superar

**TP Final · Aprendizaje de Máquina I (CEIA-FIUBA)** · Jaime Pinzón (a2629)

## ¿Por qué un baseline, y por qué estos?

Un baseline responde la pregunta *"¿cuánto se puede lograr sin aprender casi nada?"*. Si un modelo complejo no supera con claridad este piso, su costo no se justifica — esa comparación es el eje del informe final.

- **`DummyRegressor(strategy="mean")`**: predice siempre la media del train. Es el predictor constante que minimiza el error cuadrático.
- **`DummyRegressor(strategy="median")`**: predice siempre la mediana. Es el predictor constante que minimiza el **MAE** — nuestra métrica principal — *respecto de la distribución sobre la que se calcula* (matiz que resultará importante, ver más abajo).
- **Regresión lineal (OLS)**: la referencia paramétrica más simple que sí usa las features. Marca cuánta señal lineal hay en los datos.

**Nota metodológica:** la cátedra pide explícitamente NO usar regresión logística como baseline. Además de la indicación, hay una razón técnica: la regresión logística es un **clasificador** (modela probabilidades de clases discretas); nuestro problema es de **regresión** sobre días de internación, donde el análogo correcto es la regresión lineal.

Los tres baselines usan el mismo preprocesador que usarán todos los modelos (`build_preprocessor`), sin escalado: el Dummy ignora las features y OLS es invariante a transformaciones afines de las columnas.

In [1]:
import time

import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from src.config import DIR_PROCESSED, TARGET
from src.evaluacion import evaluar, registrar
from src.pipelines import build_preprocessor

X_train = pd.read_parquet(DIR_PROCESSED / "X_train.parquet")
X_test = pd.read_parquet(DIR_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(DIR_PROCESSED / "y_train.parquet")[TARGET]
y_test = pd.read_parquet(DIR_PROCESSED / "y_test.parquet")[TARGET]

print(f"train: {X_train.shape} | test: {X_test.shape}")
print(f"media del train: {y_train.mean():.3f} dias | mediana: {y_train.median():.1f} dias")

train: (53756, 8) | test: (20354, 8)
media del train: 4.087 dias | mediana: 3.0 dias


In [2]:
modelos = {
    "baseline_media": DummyRegressor(strategy="mean"),
    "baseline_mediana": DummyRegressor(strategy="median"),
    "regresion_lineal": LinearRegression(),
}
notas = {
    "baseline_media": "DummyRegressor(mean); predictor constante",
    "baseline_mediana": "DummyRegressor(median); minimiza MAE sobre la dist. del train",
    "regresion_lineal": "OLS sobre las 17 features del preprocesador comun",
}

resultados = {}
for nombre, estimador in modelos.items():
    pipe = Pipeline([
        ("preprocesador", build_preprocessor(scale=False)),
        ("modelo", estimador),
    ])
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    tiempo = time.perf_counter() - t0
    metricas = evaluar(pipe, X_test, y_test)
    metricas["tiempo_s"] = round(tiempo, 2)
    registrar(nombre, metricas, notas=notas[nombre])
    resultados[nombre] = metricas

pd.DataFrame(resultados).T.round(4)

,mae,rmse,r2,tiempo_s
baseline_media,2.2805,3.0009,-0.0107,0.09
baseline_mediana,2.2929,3.2952,-0.2187,0.05
regresion_lineal,1.8616,2.4666,0.3171,0.10


In [3]:
# El registro usa UPSERT por nombre: re-ejecutar este notebook NO duplica filas.
from src.evaluacion import RUTA_METRICAS

registrar("baseline_media", resultados["baseline_media"], notas=notas["baseline_media"])
tabla = pd.read_csv(RUTA_METRICAS)
assert tabla["modelo"].is_unique, "hay modelos duplicados en metricas.csv"
print(f"metricas.csv: {len(tabla)} filas, sin duplicados")
tabla.round(4)

metricas.csv: 7 filas, sin duplicados


,modelo,mae,rmse,r2,notas,tiempo_s
0,baseline_media,2.2805,3.0009,-0.0107,DummyRegressor(mean); predictor constante,0.09
1,baseline_mediana,2.2929,3.2952,-0.2187,DummyRegressor(median); minimiza MAE sobre la ...,0.05
2,regresion_lineal,1.8616,2.4666,0.3171,OLS sobre las 17 features del preprocesador comun,0.10
3,knn,1.8341,2.4731,0.3136,"k=80, weights=distance, p=1; grilla 32 configs...",0.11
4,svr_lineal,1.8343,2.5168,0.2891,"LinearSVR C=0.01, eps=0.5; train completo; bus...",0.30
5,svr_rbf,1.7687,2.4425,0.3304,"SVR-RBF ['C=8.598', 'epsilon=0.4397', 'gamma=0...",110.20
6,arbol_podado,1.8585,2.4829,0.3081,ccp_alpha=9.62e-04 por CV5 sobre 25 candidatos...,6.05


## ¿Por qué la media le ganó a la mediana en MAE, si "la mediana minimiza el MAE"?

La propiedad es cierta pero **relativa a la distribución sobre la que se evalúa**. Los baselines se ajustan sobre el train **filtrado por IQR** (sin estadías largas) y se evalúan sobre el test **sin filtrar**. El filtro corrió la mediana del train a 3 días, pero la mediana del test es 4: la constante 3 queda sistemáticamente lejos de la cola alta que el test sí conserva. La celda siguiente lo cuantifica.

In [4]:
# V3: evidencia numerica del fenomeno media-vs-mediana
print(f"y_train: media={y_train.mean():.4f}  mediana={y_train.median():.1f}  "
      f"skew={y_train.skew():.3f}  max={y_train.max():.0f}")
print(f"y_test : media={y_test.mean():.4f}  mediana={y_test.median():.1f}  "
      f"skew={y_test.skew():.3f}  max={y_test.max():.0f}")
print()
print(f"MAE en test prediciendo la mediana del train (3.0):   {(y_test - 3.0).abs().mean():.4f}")
print(f"MAE en test prediciendo la media del train (4.087):   {(y_test - y_train.mean()).abs().mean():.4f}")
print(f"MAE en test prediciendo la mediana del TEST (4.0):    {(y_test - y_test.median()).abs().mean():.4f}"
      "  <- optimo teorico entre constantes, inalcanzable sin mirar el test")

y_train: media=4.0869  mediana=3.0  skew=0.985  max=12
y_test : media=4.3959  mediana=4.0  skew=1.134  max=14

MAE en test prediciendo la mediana del train (3.0):   2.2929
MAE en test prediciendo la media del train (4.087):   2.2805
MAE en test prediciendo la mediana del TEST (4.0):    2.2596  <- optimo teorico entre constantes, inalcanzable sin mirar el test


## Lectura de negocio

- **El piso a superar es MAE ≈ 2,28 días** (baseline de media). La política ingenua "asumir que todo paciente se queda lo mismo que el paciente típico" se equivoca, en promedio, 2,28 días por paciente. Todo modelo de los notebooks 03–06 se juzga por cuántos días le recorta a ese piso.
- **Detalle honesto que muestra la tabla:** la media (2,2805) le ganó a la mediana (2,2929) porque ambas constantes se calculan sobre el train filtrado (mediana 3) pero se evalúan sobre el test completo (mediana 4, con 450 estadías de 13–14 días). El óptimo teórico entre constantes — predecir la mediana del test, 4 días — daría 2,2596, y ninguna constante honesta puede alcanzarlo. Es la primera evidencia visible de que evaluar sobre el test sin filtrar — como ocurriría en producción — penaliza los supuestos que solo valen en el train.
- La **regresión lineal** baja el MAE a 1,86 días (recorta 0,42 días, −18%) con R² = 0,32: hay señal lineal real en las 17 features, pero dos tercios de la varianza siguen sin explicarse — ese es el espacio que los modelos no lineales (KNN, SVR, árboles, ensambles) intentarán capturar.
- Contexto operativo: recortar 0,42 días de error medio, a escala de un hospital con miles de admisiones anuales, ya es capacidad de camas real; cada décima adicional que recorten los modelos siguientes se traduce directamente en mejor planificación de ocupación.

**Siguiente notebook (03):** KNN regressor — primer modelo real, con la justificación de por qué exige escalado.